## Challenging SRL models (A): **Logistic Regression**

In [1]:
import a1standalone
import a1calculation

Below are the relevant files that we need for this notebook.

In [2]:
# Challenge datasets
a1_1= './Data/A1.1_MFT_ARG0_SD.tsv'
a1_2= './Data/A1.2_INV_ARG0_LD.tsv'
b1= './Data/B1_DIR_predicate_subst.tsv'
c1= './Data/C1_MFT_ARG1_Cognate_object.tsv'
d1= './Data/D1_INV_ARG1_CAU_INC.tsv'
d2= './Data/D2_INV_ARG1_Voice.tsv'
e1= './Data/E1_DIR_PP_TMP_LOC.tsv'
f1= './Data/F1_MFT_ARG0_Verb_order.tsv'

#prediction files
pa1_1= './Predictions/loga11.tsv'
pa1_2= './Predictions/loga12.tsv'
pb1= './Predictions/logb1.tsv'
pc1= './Predictions/logc1.tsv'
pd1= './Predictions/logd1.tsv'
pd2= './Predictions/logd2.tsv'
pe1= './Predictions/loge1.tsv'
pf1= './Predictions/logf1.tsv'
#logreg and vectoriser
model= './Models/a1_srl_logreg.sav'
vectorizer= './Models/a1_vectorizer.sav'

### Dataset Checks

As described in the paper, the datasets are generated manually/automatically. Each case has been inspected manually. However, to prevent there being a misalignment between the length of the token and the length of predicate/argument labels per use case, automatic safe checks are also conducted.

The function below is executed to conduct length checks.

In [3]:
def precheck (infile):
    '''
    The function checks whether the length of tokens and the length of labels per use case in the file.

    Assumption: The .tsv is in a four-column format:
    0) use_case separated by space
    1) predicate; token in focus
    2) predicate labels ('_'or'x') separated by commas
    3) argument labels (argument labels or '_') separated by commas

    parameters:
    -infile: file path to the dataset

    prints：
    'Case x has a mistake, check!':if there is misalignment
    'None': if there is no mistakes
    '''
    with open (infile, 'r', encoding= 'utf-8') as f:
        lines= f.read().split('\n')

    for n, line in enumerate(lines):
        if line: #skip empty lines
            cols= line.strip().split('\t')
            case= cols[0].strip().split(' ')
            predicate= cols[2].strip().split(',')
            argument= cols[3].strip().split(',')
            if not (len(case) == len(predicate) == len(argument)):
                print(f'In {infile}, case {n+1} has a mistake, check!')
                continue
        

In [4]:
datasets= [a1_1, a1_2, b1, c1, d1, d2, e1, f1]
for file in datasets:
    precheck (file)

### Testing/Predicting

The trained logistic regression model is then tested on the challenge datasets above. This is achieved through the function below.

In [5]:
def log_challenge (infile, model, vectorizer, prediction_file):
    '''
    The function loads the trained model and vectorizer to predict the challenge dataset. 
    Then it automatically calculates the failure rate.

    Assumption: The .tsv is in a four-column format:
    0) use_case separated by space
    1) predicate; token in focus
    2) predicate labels ('_'or'x') separated by commas
    3) argument labels (argument labels or '_') separated by commas

    parameters:
    -infile: file path to the dataset
    -model: the trained LogReg
    -vectorizer: the trained  vectorizer
    -prediction_file: a tsv file to save features and predictions
    returns:
    gold: a list of lists
    pred: a list of preds
    '''
    with open (infile, 'r', encoding= 'utf-8') as f:
        lines= f.read().split('\n')

    #set counter
    god= []
    pred= []
    
    with open(prediction_file, 'w', encoding='utf-8') as output:
    
        for line in lines:
            if line: #skip empty lines
                cols= line.strip().split('\t')
                tokens= cols[0].strip().strip('"').split(' ')
                predicate= cols[2].strip().strip('"').split(',')
                argument= cols[3].strip().strip('"').split(',')
    
                #using the standalone function to predict each sentence
                feature, prediction= a1standalone.logreg_srl(tokens, predicate, model, vectorizer)
    
                
                for n, gold in enumerate(argument):
                    if gold == 'O':
                        argument[n]= '_' #For logreg, the `O` class is _, so we need to change the gold labels

                    if gold != '_':
                        target_token= tokens[n]
    
                #output the updated prediction
                output_tokens= ' '.join(tokens)
                output_gold= ','.join(argument)
                output_prediction= ','.join(prediction)
                feature_dict= str(feature)
                output.write( output_tokens + '\t' + feature_dict + '\t' + output_gold+ '\t'+ output_prediction +'\n')

                #collect gold and predictions for later use
                god.append(argument)
                pred.append(prediction)
    
    return god, pred

#### A1.1 and A1.2

In [6]:
a11gold, a11prd= log_challenge (a1_1, model, vectorizer, pa1_1)

In [7]:
a1calculation.mft_error (a11gold, a11prd)

The MFT/general error rate is 0.56. Case(s) [5, 6, 11, 21, 22, 23, 26, 28, 29, 30, 33, 35, 36, 37, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 51, 52, 54, 55, 56, 57, 59, 60, 61, 65, 66, 67, 68, 73, 74, 75, 77, 78, 79, 81, 82, 83, 85, 86, 87, 88, 93, 94, 95, 96, 97, 98] is/are wrong


In [8]:
a12gold, a12prd= log_challenge (a1_2, model, vectorizer, pa1_2)

In [9]:
a1calculation.inv_two_datasets (a11gold, a11prd, a12gold, a12prd)

The raw error rate is 0.11, with [4, 9, 27, 42, 56, 62, 64, 69, 72, 78, 100] wrong
The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.2, with [4, 9, 27, 42, 62, 64, 69, 72, 100] wrong


#### B1

In [7]:
b1gold, b1prd= log_challenge (b1, model, vectorizer, pb1)

In [8]:
a1calculation.dir_onedataset (b1gold, b1prd)

The raw error rate is 0.57, with pair [1, 4, 5, 7] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) ia 0.0, with pair [] wrong


#### C1

In [9]:
c1gold, c1prd= log_challenge (c1, model, vectorizer, pc1)

In [10]:
a1calculation.mft_error (c1gold, c1prd)

The MFT/general error rate is 1.0. Case(s) [1, 2, 3, 4, 5, 6, 7, 8] is/are wrong


#### D1, D2

In [19]:
d1gold, d1prd= log_challenge (d1, model, vectorizer, pd1)

In [20]:
a1calculation.inv_onedataset (d1gold, d1prd)

The raw error rate is 0.16, with pair [1, 5, 24, 33, 35, 37, 46, 48] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.38, with pair [1, 24, 35, 37, 46] wrong


In [21]:
d2gold, d2prd= log_challenge (d2, model, vectorizer, pd2)

In [22]:
a1calculation.inv_onedataset (d2gold, d2prd)

The raw error rate is 0.36, with pair [1, 2, 4, 5, 8, 10, 11, 12, 22, 24, 25, 28, 32, 35, 36, 38, 48, 49] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) is 0.73, with pair [1, 2, 4, 8, 10, 11, 12, 22, 24, 25, 28, 32, 35, 36, 38, 49] wrong


#### E1

In [23]:
e1gold, e1prd= log_challenge (e1, model, vectorizer, pe1)

In [24]:
a1calculation.dir_onedataset (e1gold, e1prd)

The raw error rate is 0.0, with pair [] wrong

The conditional error rate (i.e. cases where starting sentences are correctly predicted) ia 0.0, with pair [] wrong


#### F1

In [26]:
f1gold, f1prd= log_challenge (f1, model, vectorizer, pf1)

In [27]:
a1calculation.mft_error (f1gold, f1prd)

The MFT/general error rate is 0.62. Case(s) [2, 4, 5, 7, 8] is/are wrong
